# 02 — Susceptibility Scoring

Computes the **Flood Susceptibility Index** using weighted multi-criteria overlay.

| Factor | Weight | Data status |
|---|---:|---|
| Ground elevation | 40% | Real (SRTM via QGIS) |
| Monsoon rainfall | 25% | Estimated (IMD reports) |
| Imperviousness | 20% | Estimated |
| Proximity to water | 15% | Estimated |

This is a standard, transparent GIS methodology used in published flood-risk studies.
Every zone's score is fully traceable back to its input factors.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

zones = [
  {
    "zone_no": 1,
    "zone_name": "Thiruvottiyur",
    "region": "North",
    "elevation_m": 5.272,
    "susceptibility_score": 82.4,
    "risk_class": "Very High"
  },
  {
    "zone_no": 2,
    "zone_name": "Manali",
    "region": "North",
    "elevation_m": 4.557,
    "susceptibility_score": 50.6,
    "risk_class": "Moderate"
  },
  {
    "zone_no": 3,
    "zone_name": "Madhavaram",
    "region": "North",
    "elevation_m": 10.313,
    "susceptibility_score": 28.4,
    "risk_class": "Low"
  },
  {
    "zone_no": 4,
    "zone_name": "Tondiarpet",
    "region": "North",
    "elevation_m": 6.442,
    "susceptibility_score": 79.5,
    "risk_class": "Very High"
  },
  {
    "zone_no": 5,
    "zone_name": "Royapuram",
    "region": "North",
    "elevation_m": 8.612,
    "susceptibility_score": 72.9,
    "risk_class": "High"
  },
  {
    "zone_no": 6,
    "zone_name": "Thiru-Vi-Ka Nagar",
    "region": "Central",
    "elevation_m": 9.593,
    "susceptibility_score": 53.3,
    "risk_class": "Moderate"
  },
  {
    "zone_no": 7,
    "zone_name": "Ambattur",
    "region": "Central",
    "elevation_m": 16.376,
    "susceptibility_score": 13.3,
    "risk_class": "Low"
  },
  {
    "zone_no": 8,
    "zone_name": "Anna Nagar",
    "region": "Central",
    "elevation_m": 12.595,
    "susceptibility_score": 35.5,
    "risk_class": "Moderate"
  },
  {
    "zone_no": 9,
    "zone_name": "Teynampet",
    "region": "Central",
    "elevation_m": 11.14,
    "susceptibility_score": 69.4,
    "risk_class": "High"
  },
  {
    "zone_no": 10,
    "zone_name": "Kodambakkam",
    "region": "Central",
    "elevation_m": 13.136,
    "susceptibility_score": 41.5,
    "risk_class": "Moderate"
  },
  {
    "zone_no": 11,
    "zone_name": "Valasaravakkam",
    "region": "South",
    "elevation_m": 14.341,
    "susceptibility_score": 13.7,
    "risk_class": "Low"
  },
  {
    "zone_no": 12,
    "zone_name": "Alandur",
    "region": "South",
    "elevation_m": 12.425,
    "susceptibility_score": 46.7,
    "risk_class": "Moderate"
  },
  {
    "zone_no": 13,
    "zone_name": "Adyar",
    "region": "South",
    "elevation_m": 9.174,
    "susceptibility_score": 79.7,
    "risk_class": "Very High"
  },
  {
    "zone_no": 14,
    "zone_name": "Perungudi",
    "region": "South",
    "elevation_m": 4.025,
    "susceptibility_score": 86.2,
    "risk_class": "Very High"
  },
  {
    "zone_no": 15,
    "zone_name": "Sholinganallur",
    "region": "South",
    "elevation_m": 3.265,
    "susceptibility_score": 87.1,
    "risk_class": "Very High"
  }
]
df = pd.DataFrame(zones)

# Additional factor estimates
rainfall = {1:1280,2:1120,3:1090,4:1260,5:1240,6:1180,7:1100,
            8:1150,9:1290,10:1170,11:1080,12:1200,13:1320,14:1310,15:1300}
imperv   = {1:0.75,2:0.45,3:0.55,4:0.79,5:0.80,6:0.74,7:0.62,
            8:0.76,9:0.85,10:0.78,11:0.60,12:0.70,13:0.82,14:0.71,15:0.68}
dist_w   = {1:1.2,2:1.8,3:3.5,4:1.0,5:0.6,6:2.5,7:3.8,
            8:4.2,9:1.5,10:3.0,11:4.5,12:2.0,13:0.8,14:1.5,15:1.2}

df['rainfall_mm']    = df.zone_no.map(rainfall)
df['imperviousness'] = df.zone_no.map(imperv)
df['dist_water_km']  = df.zone_no.map(dist_w)

# Normalise (1 = max flood contribution)
def norm(x): return (x - x.min()) / (x.max() - x.min())

df['f_elev']   = 1 - norm(df.elevation_m)   # lower = riskier
df['f_rain']   = norm(df.rainfall_mm)
df['f_imperv'] = norm(df.imperviousness)
df['f_dist']   = 1 - norm(df.dist_water_km) # closer = riskier

# Weighted overlay
W = dict(elev=0.40, rain=0.25, imperv=0.20, dist=0.15)
df['score'] = (W['elev']*df.f_elev + W['rain']*df.f_rain +
               W['imperv']*df.f_imperv + W['dist']*df.f_dist) * 100

print('Weights:', W)
print(f'Score range: {df.score.min():.1f} — {df.score.max():.1f}')
print()
print(df[['zone_name','elevation_m','score','risk_class']].sort_values('score',ascending=False).to_string(index=False))

## Score Distribution

In [ ]:
df_s = df.sort_values('score')
colors = {'Very High':'#c62828','High':'#ef6c00','Moderate':'#f9a825','Low':'#388e3c'}
bar_colors = [colors[r] for r in df_s.risk_class]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(df_s.zone_name, df_s.score, color=bar_colors)
for bar, s in zip(bars, df_s.score):
    ax.text(s+0.5, bar.get_y()+bar.get_height()/2, f'{s:.1f}', va='center', fontsize=8)
ax.set_xlabel('Flood Susceptibility Score (0-100)')
ax.set_title('Chennai Zones — Flood Susceptibility Index')
plt.tight_layout()
plt.savefig('../assets/chennai_susceptibility_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to assets/')